In [ ]:
# ==============================================================================
# CELL 1: CONFIGURATION & SECRETS SETUP
# ==============================================================================
# Top-level configuration variables
REEL_COUNT = 2                 # Number of autonomous morning reels to generate (1, 2, or 3)
SCENE_PRESET = "auto"          # "auto" (rotating 4 sets: diner, living_room, desk, rooftop)
INSTA_URL = ""                 # Optional: paste an Instagram Reel or YouTube Short URL to recreate
REPO_SLUG = "maazmdx/unlimited-reels-engine"
BRANCH = "main"

import os
import sys
import json
from pathlib import Path

print("🔑 [Cell 1] Loading secrets and setting environment variables...")

# 1. First check attached dataset file /kaggle/input/reels-engine-secrets/secrets.json
secrets_file = Path("/kaggle/input/reels-engine-secrets/secrets.json")
loaded_from_dataset = False

if secrets_file.exists():
    try:
        with open(secrets_file, "r", encoding="utf-8") as f:
            sec_data = json.load(f)
        for key in ["GEMINI_API_KEY", "GROQ_API_KEY", "TELEGRAM_BOT_TOKEN", "TELEGRAM_CHAT_ID", "GDRIVE_FOLDER_ID", "GITHUB_TOKEN", "OPENROUTER_API_KEY", "MISTRAL_API_KEY", "LLM_GATEWAY_URL", "LLM_GATEWAY_KEY"]:
            if sec_data.get(key):
                os.environ[key] = str(sec_data[key]).strip()
                print(f"✅ {key} loaded from dataset secrets.json")
        if "GDRIVE_CREDENTIALS" in sec_data:
            gcreds = sec_data["GDRIVE_CREDENTIALS"]
            if isinstance(gcreds, (dict, list)):
                os.environ["GDRIVE_CREDENTIALS"] = json.dumps(gcreds)
            else:
                os.environ["GDRIVE_CREDENTIALS"] = str(gcreds).strip()
            print("✅ GDRIVE_CREDENTIALS loaded from dataset secrets.json")
        loaded_from_dataset = True
    except Exception as e:
        print(f"⚠️ Error reading dataset secrets.json: {e}")

# 2. Fall back to Kaggle UserSecretsClient if dataset file is absent
if not loaded_from_dataset:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        
        # 1. Gemini API Key (Required for AI screenplay & multimodal analysis)
        try:
            os.environ["GEMINI_API_KEY"] = secrets.get_secret("GEMINI_API_KEY")
            print("✅ GEMINI_API_KEY loaded from Kaggle Secrets")
        except Exception:
            print("⚠️ GEMINI_API_KEY not found in Kaggle Secrets (checking os.environ)")
            
        # 1b. Groq API Key (High-Speed Zero-Fail Fallback for LLM Screenplays)
        try:
            os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")
            print("✅ GROQ_API_KEY loaded from Kaggle Secrets")
        except Exception:
            pass
            
        # 2. GitHub Token (Optional: only needed if repository is private)
        try:
            os.environ["GITHUB_TOKEN"] = secrets.get_secret("GITHUB_TOKEN")
            print("✅ GITHUB_TOKEN loaded from Kaggle Secrets")
        except Exception:
            pass

        # 3. Google Drive Credentials & Folder (Optional: for cloud delivery)
        try:
            os.environ["GDRIVE_CREDENTIALS"] = secrets.get_secret("GDRIVE_CREDENTIALS")
            print("✅ GDRIVE_CREDENTIALS loaded from Kaggle Secrets")
        except Exception:
            pass
            
        try:
            os.environ["GDRIVE_FOLDER_ID"] = secrets.get_secret("GDRIVE_FOLDER_ID")
            print("✅ GDRIVE_FOLDER_ID loaded from Kaggle Secrets")
        except Exception:
            pass
            
        for opt_key in ["TELEGRAM_BOT_TOKEN", "TELEGRAM_CHAT_ID", "OPENROUTER_API_KEY", "MISTRAL_API_KEY", "LLM_GATEWAY_URL", "LLM_GATEWAY_KEY"]:
            try:
                val = secrets.get_secret(opt_key)
                if val:
                    os.environ[opt_key] = str(val).strip()
                    print(f"✅ {opt_key} loaded from Kaggle Secrets")
            except Exception:
                pass

    except ImportError:
        print("ℹ️ Running outside Kaggle environment; using existing os.environ variables.")

# Verify mandatory secret
if not os.environ.get("GEMINI_API_KEY"):
    print("⚠️ WARNING: GEMINI_API_KEY is unset! Screenplays will use curated fallback matrix.")
else:
    print("🚀 API authentication verified.")

print(f"\n📋 Configuration Summary:")
print(f"  • Reel Count   : {REEL_COUNT}")
print(f"  • Scene Preset : {SCENE_PRESET}")
print(f"  • Custom URL   : {INSTA_URL if INSTA_URL else '(None - running morning batch)'}")
print(f"  • Repository   : {REPO_SLUG} ({BRANCH})")


In [ ]:
# ==============================================================================
# CELL 2: CLONE OR PULL LATEST REPOSITORY
# ==============================================================================
import os
import subprocess
from pathlib import Path

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR = WORKING_DIR / "unlimited-reels-engine"

github_token = os.environ.get("GITHUB_TOKEN", "").strip()
if github_token:
    CLONE_URL = f"https://{github_token}@github.com/{REPO_SLUG}.git"
else:
    CLONE_URL = f"https://github.com/{REPO_SLUG}.git"

if not REPO_DIR.exists():
    print(f"📥 Cloning repository {REPO_SLUG} into {REPO_DIR}...")
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, CLONE_URL, str(REPO_DIR)], check=True)
    print("✅ Repository cloned successfully.")
else:
    print(f"🔄 Repository exists at {REPO_DIR}. Pulling latest changes...")
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=str(REPO_DIR), check=True)
    print("✅ Repository up to date.")

os.chdir(str(REPO_DIR))
print(f"📂 Current Working Directory: {os.getcwd()}")

In [ ]:
# ==============================================================================
# CELL 3: RUNTIME DEPENDENCY INSTALLATION (Node 20, FFmpeg, Chrome libs, npm, pip)
# ==============================================================================
import os
import sys
import shutil
import subprocess

print("🛠️ [Cell 3] Setting up fast headless render environment on Kaggle T4...")

# 1. Verify / Install Node.js 20 LTS
need_node = True
node_bin = shutil.which("node")
if node_bin:
    try:
        ver_res = subprocess.run(["node", "-v"], capture_output=True, text=True)
        major_ver = int(ver_res.stdout.strip().lstrip("v").split(".")[0])
        if major_ver >= 20:
            print(f"✅ Node.js {ver_res.stdout.strip()} is already installed.")
            need_node = False
        else:
            print(f"⚠️ Found Node.js {ver_res.stdout.strip()} (< 20). Upgrading to Node 20 LTS...")
    except Exception:
        pass

if need_node:
    print("📦 Installing official Node.js 20 LTS standalone binary (bypassing apt locks)...")
    node_tar_cmd = (
        "curl -fsSL https://nodejs.org/dist/v20.18.0/node-v20.18.0-linux-x64.tar.xz "
        "| tar -xJ -C /usr/local --strip-components=1"
    )
    subprocess.run(node_tar_cmd, shell=True, check=True)
    ver_res = subprocess.run(["node", "-v"], capture_output=True, text=True)
    print(f"✅ Installed Node.js: {ver_res.stdout.strip()}")

# 2. Install FFmpeg and essential Chromium headless runtime libraries
print("📦 Checking system multimedia and Chromium dependencies...")
apt_cmd = (
    "apt-get update -qq && apt-get install -y -qq "
    "ffmpeg libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 "
    "libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 "
    "libgbm1 libasound2 || true"
)
subprocess.run(apt_cmd, shell=True)
print("✅ System libraries verified.")

# 3. Install Python dependencies
print("📦 Installing Python dependencies from requirements.txt...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Python dependencies installed.")

# 3b. Optional GPU TTS Acceleration (Chatterbox-Turbo on Kaggle CUDA)
try:
    import torch
    if torch.cuda.is_available():
        print("🔥 CUDA GPU detected. Installing transformers==4.46.3 & chatterbox-tts...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools<81", "huggingface-hub==0.36.2", "transformers==4.46.3", "diffusers<0.33", "resemble-perth", "conformer", "s3tokenizer", "omegaconf"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "chatterbox-tts"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", "huggingface-hub==0.36.2"], check=True)
except Exception as e:
    print(f"ℹ️ Chatterbox GPU optional install skipped: {e}")

# 4. Install Remotion Dependencies & Ensure Headless Browser
remotion_dir = REPO_DIR / "my-edu-reels"
print(f"📦 Installing Remotion npm packages in {remotion_dir}...")
subprocess.run(["npm", "ci", "--prefer-offline", "--no-audit"], cwd=str(remotion_dir), check=True)

print("🌐 Downloading / verifying Remotion headless Chromium browser...")
subprocess.run(["npx", "remotion", "browser", "ensure"], cwd=str(remotion_dir), check=True)
print("✅ Remotion headless environment ready.")

# 5. Check GPU acceleration (NVIDIA T4) & Chatterbox GPU Voice Engine
gpu_res = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_res.returncode == 0:
    first_line = [l for l in gpu_res.stdout.splitlines() if "NVIDIA" in l or "Tesla" in l or "T4" in l]
    print(f"🔥 GPU Acceleration Active: {first_line[0] if first_line else 'NVIDIA GPU detected'}")
    subprocess.run([
        sys.executable, "-c",
        "import perth; from chatterbox.tts import ChatterboxTTS; assert perth.PerthImplicitWatermarker is not None; print('✅ Chatterbox GPU + Perth ready')"
    ], check=True)
else:
    print("ℹ️ Running in CPU mode (multi-threaded SwiftShader).")


In [ ]:
# ==============================================================================
# CELL 4: EXECUTE VIDEO GENERATION (LIVE PROGRESS STREAMING)
# ==============================================================================
import os
import sys
import subprocess

print("🎬 [Cell 4] Launching Autonomous Video Production Pipeline...")
os.environ["PYTHONUNBUFFERED"] = "1"

if INSTA_URL.strip():
    print(f"🎯 Running Single Reel Recreation for: {INSTA_URL}")
    cmd = [
        sys.executable,
        "engine/master_orchestrator.py",
        "--insta", INSTA_URL.strip()
    ]
else:
    print(f"🌅 Running Morning Batch: {REEL_COUNT} reels, preset: '{SCENE_PRESET}'")
    cmd = [
        sys.executable,
        "engine/morning_batch_runner.py",
        "--count", str(REEL_COUNT),
        "--scene-preset", SCENE_PRESET,
        "--telegram"
    ]

# Execute process with real-time stdout/stderr streaming
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

for line in process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()

ret_code = process.wait()

if ret_code != 0:
    raise RuntimeError(f"❌ Pipeline failed with exit code {ret_code}")
else:
    print("\n🎉 [Cell 4] Video production batch completed successfully!")

In [ ]:
# ==============================================================================
# CELL 5: EXPORT DELIVERED REELS & RENDER SUMMARY
# ==============================================================================
import os
import shutil
import json
from pathlib import Path

DELIVERED_DIR = WORKING_DIR / "delivered_reels"
DELIVERED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = REPO_DIR / "out"

print(f"📦 [Cell 5] Exporting generated reels to {DELIVERED_DIR}...")

# Locate newest morning batch or reels
delivered_files = []
batch_folders = sorted(OUT_DIR.glob("morning_*"), key=lambda p: p.stat().st_mtime, reverse=True)

if batch_folders:
    latest_batch = batch_folders[0]
    print(f"📂 Found latest batch directory: {latest_batch.name}")
    for item in latest_batch.iterdir():
        dest = DELIVERED_DIR / item.name
        if item.is_dir():
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(item, dest)
        else:
            if os.path.realpath(str(item)) != os.path.realpath(str(dest)):
                shutil.copy2(item, dest)
        delivered_files.append(dest)
else:
    # Look for any recent .mp4 and json in out/
    for mp4 in OUT_DIR.glob("*.mp4"):
        dest = DELIVERED_DIR / mp4.name
        if os.path.realpath(str(mp4)) != os.path.realpath(str(dest)):
            shutil.copy2(mp4, dest)
        delivered_files.append(dest)

print("\n" + "=" * 80)
print("📊 PRODUCTION BATCH SUMMARY & DELIVERED ASSETS")
print("=" * 80)

reels_found = list(DELIVERED_DIR.glob("*.mp4"))
if not reels_found:
    print("⚠️ No MP4 videos found in delivered directory.")
else:
    print(f"| {'Filename':<38} | {'Size (MB)':<10} | {'Format':<12} |")
    print(f"|{'-'*40}|{'-'*12}|{'-'*14}|")
    for r in reels_found:
        size_mb = r.stat().st_size / (1024 * 1024)
        print(f"| {r.name:<38} | {size_mb:<10.2f} | {'9:16 (1080p)':<12} |")

print("\n📁 Output Location for Download:")
print(f"   {DELIVERED_DIR}")
print("\n💡 In Kaggle:")
print("   Click the 'Data' tab in the right-side panel, expand '/kaggle/working/delivered_reels',")
print("   and click the three dots next to any .mp4 file to download directly!")
print("\n✅ Batch complete. Assets delivered to Telegram via morning runner.")
print("=" * 80)
